# A TIFF volume to OME-Zarr, and into Neuroglancer

**Dataset.** The head of an Argentine horned frog, *Ceratophrys ornata*,
iodine-stained micro-CT. It is the volume the voxels session renders, so this
converts something you have already looked at. Source and licence: `example_data/ceratophrys-ornata/CREDITS.md` (CC0).

In [12]:
import json
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from urllib.parse import quote

import ngff_zarr as nz
import numpy as np
import tifffile as tiff

OUT = Path("results") / "ngff"
OUT.mkdir(parents=True, exist_ok=True)


def directory_size(path):
    """Bytes on disk under `path`."""
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())


def mb(n):
    return f"{n / 1e6:.2f} MB"

## 1. Load the TIFF, and its voxel size

In [13]:
VOLUME = Path("../example_data/ceratophrys-ornata/frog-head-256x256x195.tif")
VOXEL_UM = 202.0          # isotropic, from CREDITS.md

vol = tiff.imread(VOLUME)

print(f"shape  {vol.shape}  (z, y, x)")
print(f"dtype  {vol.dtype}, range {vol.min()}-{vol.max()}")
print(f"extent {np.round(np.array(vol.shape) * VOXEL_UM / 1000, 1)} mm")
print(f"TIFF on disk: {mb(VOLUME.stat().st_size)}")

shape  (195, 256, 256)  (z, y, x)
dtype  uint8, range 0-255
extent [39.4 51.7 51.7] mm
TIFF on disk: 3.08 MB


## 2. Build the multiscale pyramid

A pyramid is the same volume at several resolutions. A viewer showing the
whole specimen reads the smallest level; zoom in and it reads chunks of a
finer one, only for the part you are looking at.

In [14]:
image = nz.to_ngff_image(
    vol,
    dims=("z", "y", "x"),
    scale={"z": VOXEL_UM, "y": VOXEL_UM, "x": VOXEL_UM},
    axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"},
    name="frog-head",
)

multiscales = nz.to_multiscales(image, scale_factors=[2, 4, 8], chunks=64)

print(f"{'level':<7}{'shape':<22}{'voxels':>12}")
for i, level in enumerate(multiscales.images):
    print(f"{i:<7}{str(level.data.shape):<22}{level.data.size:>12,}")

total = sum(level.data.size for level in multiscales.images)
print(f"\npyramid is {total / multiscales.images[0].data.size:.3f}x "
      f"the voxels of level 0")

level  shape                       voxels
0      (195, 256, 256)         12,779,520
1      (97, 128, 128)           1,589,248
2      (48, 64, 64)               196,608
3      (24, 32, 32)                24,576

pyramid is 1.142x the voxels of level 0


## 3. Write the OME-Zarr

One call. `version="0.4"` is the version the widest set of tools reads today;
0.5 exists and moves the metadata into zarr v3.

In [15]:
STORE = OUT / "frog-head.ome.zarr"
shutil.rmtree(STORE, ignore_errors=True)

nz.to_ngff_zarr(STORE, multiscales, version="0.4")

print(f"wrote {STORE}")
print(f"on disk: {mb(directory_size(STORE))}  "
      f"(TIFF was {mb(VOLUME.stat().st_size)})")

wrote results/ngff/frog-head.ome.zarr
on disk: 3.51 MB  (TIFF was 3.08 MB)


Print information about the output:

In [16]:
import zarr

attrs = json.loads((STORE / ".zattrs").read_text())
MULTISCALES = attrs["multiscales"][0]

group = zarr.open_group(STORE, mode="r")
print(group.tree())

/
├── scale0
│   └── frog-head (195, 256, 256) uint8
├── scale1
│   └── frog-head (97, 128, 128) uint8
├── scale2
│   └── frog-head (48, 64, 64) uint8
└── scale3
    └── frog-head (24, 32, 32) uint8



In [17]:
print(group[MULTISCALES["datasets"][0]["path"]].info_complete())

Type               : Array
Zarr format        : 2
Data type          : UInt8()
Fill value         : 0
Shape              : (195, 256, 256)
Chunk shape        : (64, 64, 64)
Order              : C
Read-only          : True
Store type         : LocalStore
Filters            : ()
Compressors        : (Zstd(level=0),)
No. bytes          : 12779520 (12.2M)
No. bytes stored   : 2997045 (2.9M)
Storage ratio      : 4.3
Chunks Initialized : 60


## 4. Validate it

The slides point at the [OME-NGFF validator](https://ome.github.io/ome-ngff-validator/),
a web page you paste a URL into. It checks the metadata against the published
JSON schema, and `ngff-zarr` bundles those schemas so you can run the same
check offline.

In [20]:
# nz.validate raises jsonschema's ValidationError, not the ngff_zarr class of
# the same name - that one belongs to its separate structural check.
import jsonschema

nz.validate(attrs, version="0.4", model="image")
print("valid OME-NGFF 0.4")

valid OME-NGFF 0.4


## 5. Serve it with CORS

A browser refuses to read data from another origin unless the server says it
may, with an `Access-Control-Allow-Origin` header. Without it Neuroglancer
stays empty while `curl` works perfectly - which is the single most common
reason this does not work first time.

`example_data/server.py` is Google's own CORS server from the Neuroglancer
repository.

In [21]:
PORT = 8123

server = subprocess.Popen(
    [sys.executable, "../example_data/server.py", "-d", str(OUT), "-p", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(2)

base = f"http://127.0.0.1:{PORT}/{STORE.name}"
with urllib.request.urlopen(f"{base}/.zattrs", timeout=10) as response:
    cors = response.headers.get("Access-Control-Allow-Origin")
print(f"serving {OUT} on port {PORT}")
print(f"Access-Control-Allow-Origin: {cors}")

chunk = f"{base}/{MULTISCALES['datasets'][0]['path']}/0/0/0"
with urllib.request.urlopen(chunk, timeout=10) as response:
    print(f"one chunk: {len(response.read())} bytes from {chunk}")

serving results/ngff on port 8123
Access-Control-Allow-Origin: *
one chunk: 298 bytes from http://127.0.0.1:8123/frog-head.ome.zarr/scale0/frog-head/0/0/0


## 6. Open it in Neuroglancer

Neuroglancer's whole state - layers, sources, camera - lives in the URL
fragment as JSON. So a link is something you build, not something you click
your way to and copy.

`zarr://` in front of an ordinary http URL is what tells it the source is
Zarr. Neuroglancer reads OME-Zarr 0.4 and 0.5, and picks up the pyramid and
the voxel size from the metadata written in step 3.

In [22]:
NEUROGLANCER = "https://neuroglancer-demo.appspot.com/#!"

state = {
    "layers": [{
        "type": "image",
        "name": "frog-head",
        "source": f"zarr://{base}/",
    }],
    "layout": "4panel",
}

url = NEUROGLANCER + quote(json.dumps(state, separators=(",", ":")), safe="")
print(url)

https://neuroglancer-demo.appspot.com/#!%7B%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22name%22%3A%22frog-head%22%2C%22source%22%3A%22zarr%3A%2F%2Fhttp%3A%2F%2F127.0.0.1%3A8123%2Ffrog-head.ome.zarr%2F%22%7D%5D%2C%22layout%22%3A%224panel%22%7D


Open that while the server in step 5 is still running.

**Chrome 142 and later will block it unless you allow it.** A page on a public
https origin reading from `127.0.0.1` needs Local Network Access permission;
allow it when the prompt appears. With no prompt the request fails as
`ERR_BLOCKED_BY_LOCAL_NETWORK_ACCESS_CHECKS`, and the giveaway is that the
browser's network tab shows the request while your server log shows nothing.

In [23]:
# Stop the server when you are done looking.
server.terminate()
server.wait(timeout=10)
print("server stopped")

server stopped


## 7. The same store, in everything else

- **napari** - `napari --plugin napari-ome-zarr results/ngff/frog-head.ome.zarr`
- **MoBIE**, **BigVolumeBrowser** - point them at the directory or its URL
- **Back into Python** - `nz.from_ngff_zarr(STORE)` gives the levels as dask
  arrays, so a level larger than memory is still something you can slice

## What to change for a volume that is actually large

This frog is 12 M voxels, which fits in memory and converts in seconds. Three
things change when it does not:

- **Chunk size.** 64³ is a reasonable default. Larger chunks mean fewer
  requests and more wasted bytes per request; smaller means the opposite
- **Levels.** Enough that the top level is a few hundred voxels across, so the
  whole-specimen view is one or two chunks
- **Reading.** Hand `to_multiscales` a dask array rather than a numpy one and
  nothing has to be in memory at once

And if the input is a vendor format rather than a TIFF, convert with
[bioformats2raw](https://github.com/glencoesoftware/bioformats2raw).